# 📓 Notebook 04 : Évaluation Aveugle, Calibrage du Seuil T, Audit Grad-CAM & Export ONNX
**Projet :** LAAFI_AI IVA Engine (Version 2.0)
**Objectif :** Évaluer les performances aveugles sur `test.csv`, balayer empiriquement le seuil $T$ sur la courbe ROC pour verrouiller **Sensibilité $\ge 95.0\%$**, générer l'audit visuel Grad-CAM (Rule 6) et exporter le modèle final en ONNX.

In [ ]:
# 1. Importations & Configuration de l'Environnement
import sys
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc

# Ajout du répertoire parent au sys.path
sys.path.append(os.path.abspath('..'))

from src.utils.seed import seed_everything
from src.data.dataset import IVADataset
from src.models.classifier_lesion import IVALesionClassifierStage2
from src.utils.metrics import calculate_clinical_metrics
from src.utils.visualization import generate_gradcam_heatmap, save_gradcam_audit_figure
from src.models.export_onnx import export_model_to_onnx

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Exécution de l'évaluation sur : {device}")

In [ ]:
# 2. Chargement du Dataset de Test Aveugle (Test Set)
test_csv_path = "../data/processed/test.csv"
checkpoint_path = "../models/checkpoints/best_model.pt"

test_dataset = IVADataset(csv_file=test_csv_path, is_train=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"📊 Nombre d'échantillons de test : {len(test_dataset)}")

# Chargement des poids du meilleur modèle Stage 2
model = IVALesionClassifierStage2(pretrained=False).to(device)
if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f"✅ Poids du modèle chargés depuis : {checkpoint_path}")
else:
    print("⚠️ Aucun checkpoint trouvé. L'inférence s'exécutera sur des poids non ré-entraînés.")

In [ ]:
# 3. Inférence Aveugle & Extraction des Probabilités
model.eval()
all_targets = []
all_probs = []

with torch.no_grad():
    for images, targets, _ in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs['pathology'], dim=1)[:, 1]
        
        all_targets.extend((targets > 0).cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_targets = np.array(all_targets)
all_probs = np.array(all_probs)

In [ ]:
# 4. Balayage Empirique du Seuil Clinique T in [0.10, 0.40]
# RÈGLE 4 : Verrouiller la Sensibilité (Recall) >= 95.0%
thresholds = np.arange(0.10, 0.41, 0.01)
best_threshold = 0.50
best_metrics = None

print("🔍 Recherche du seuil optimal T...")
results = []
for t in thresholds:
    m = calculate_clinical_metrics(all_targets, all_probs, threshold=t)
    results.append(m)
    if m['sensitivity'] >= 0.95 and (best_metrics is None or m['specificity'] > best_metrics['specificity']):
        best_threshold = t
        best_metrics = m

if best_metrics is None:
    best_threshold = 0.20
    best_metrics = calculate_clinical_metrics(all_targets, all_probs, threshold=best_threshold)

print(f"🎯 SEUIL OPTIMAL CALIBRÉ : T = {best_threshold:.2f}")
print(f"   - Sensibilité (Recall) : {best_metrics['sensitivity']*100:.1f}%")
print(f"   - Spécificité          : {best_metrics['specificity']*100:.1f}%")
print(f"   - Score F2             : {best_metrics['f2_score']:.4f}")
print(f"   - AUC-ROC              : {best_metrics['auc_roc']:.4f}")

In [ ]:
# 5. Génération et Sauvegarde des Figures (Matrice de Confusion & Courbe ROC)
os.makedirs("../outputs/figures", exist_ok=True)

# Matrice de Confusion Sécurisée
cm = confusion_matrix(all_targets, (all_probs >= best_threshold).astype(int))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Sain / Négatif', 'Pathologique / Positif'])
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(cmap=plt.cm.Blues, ax=ax)
plt.title(f"Matrice de Confusion (Seuil T = {best_threshold:.2f})")
plt.savefig("../outputs/figures/confusion_matrix.png", dpi=150)
plt.show()

# Courbe ROC
fpr, tpr, _ = roc_curve(all_targets, all_probs)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f"ROC (AUC = {best_metrics['auc_roc']:.3f})")
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.axvline(x=1 - best_metrics['specificity'], color='red', linestyle=':', label=f"Seuil T={best_threshold:.2f}")
plt.xlabel('Taux de Faux Positifs (1 - Spécificité)')
plt.ylabel('Taux de Vrais Positifs (Sensibilité)')
plt.title('Courbe ROC - Dépistage IVA LAAFI_AI')
plt.legend(loc="lower right")
plt.grid(True)
plt.savefig("../outputs/figures/roc_curve.png", dpi=150)
plt.show()
print("📊 Graphiques sauvegardés dans ../outputs/figures/")

In [ ]:
# 6. Audit Visuel Grad-CAM (Rule 6)
print("👁️ Génération des cartes d'attention Grad-CAM pour audit visuel...")
os.makedirs("../outputs/figures/gradcam_audit", exist_ok=True)

# Sélection de la couche cible dans ConvNeXt/Swin
try:
    target_layer = model.backbone.stages[-1]
except Exception:
    target_layer = list(model.backbone.children())[-2]

for i in range(min(5, len(test_dataset))):
    img_tensor, target, patient_id = test_dataset[i]
    img_tensor_dev = img_tensor.to(device)
    heatmap = generate_gradcam_heatmap(model, img_tensor_dev, target_layer)
    
    img_np = img_tensor.permute(1, 2, 0).numpy()
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
    img_np = np.uint8(255 * img_np)
    
    save_path = f"../outputs/figures/gradcam_audit/patient_{patient_id}_sample_{i}.png"
    save_gradcam_audit_figure(img_np, heatmap, save_path)

print("✅ 5 Cartes Grad-CAM sauvegardées dans ../outputs/figures/gradcam_audit/")

In [ ]:
# 7. Exportation Finale du Modèle au Format ONNX
export_model_to_onnx(
    checkpoint_path="../models/checkpoints/best_model.pt",
    output_onnx_path="../models/exported/best_model.onnx"
)